# Parameter-Efficient Fine-Tuning (PEFT) with NeMo AutoModel

This tutorial walks through **LoRA-based parameter-efficient fine-tuning**
of a pre-trained language model using
[NeMo AutoModel](https://github.com/NVIDIA-NeMo/Automodel).

We will fine-tune **Llama-3.2-1B** on the
[SQuAD](https://huggingface.co/datasets/rajpurkar/squad) dataset using
LoRA adapters -- training only a small fraction of the parameters while
keeping the base model frozen.

## Learning Goals

**What is PEFT / LoRA?**

Low-Rank Adaptation (LoRA) is a parameter-efficient fine-tuning technique
that **freezes the pre-trained model weights** and injects small, trainable
low-rank matrices into each layer. Instead of updating all *N* parameters,
LoRA trains two small matrices *A* (d x r) and *B* (r x d) where *r* is
much smaller than *d*, so the number of trainable parameters is a tiny
fraction of the full model.

**Benefits of PEFT:**

* **Much less GPU memory** -- optimizer states are only maintained for
  the small adapter parameters, not the full model.
* **Faster training** -- fewer parameters to update means faster backward
  passes.
* **Easy adapter swapping** -- you can train multiple LoRA adapters for
  different tasks and swap them at inference time without reloading the
  base model.
* **Single-GPU friendly** -- a 1B model with LoRA fits comfortably on one
  GPU, whereas full SFT of larger models may require tensor parallelism.

**Tradeoff:**

For the same amount of training data, full SFT typically achieves slightly
higher quality than LoRA. However, the gap narrows as you increase the
LoRA rank or train on more data. PEFT is the recommended starting point
when GPU resources are limited or when you need to maintain multiple
task-specific adapters.

**Prerequisites:**

* **NeMo AutoModel** installed (see the
  [AutoModel README](https://github.com/NVIDIA-NeMo/Automodel) for
  installation instructions).
* **Hugging Face token** for gated model access (Llama). Run `huggingface-cli login` or set the `HF_TOKEN` environment variable.

---
## Step 1. Prepare the Data

We use the same SQuAD dataset as the [SFT tutorial](sft.ipynb). The
built-in `make_squad_dataset` helper handles tokenization and loss
masking automatically.

In [ ]:
from datasets import load_dataset

ds = load_dataset("rajpurkar/squad", split="train")
print(f"Training samples: {len(ds)}")
print("\nExample:")
print(f"  Context: {ds[0]['context'][:200]}...")
print(f"  Question: {ds[0]['question']}")
print(f"  Answer: {ds[0]['answers']['text'][0]}")

---
## Step 2. Configure PEFT

The PEFT configuration is almost identical to SFT with three key
differences:

1. A `peft:` section defines the LoRA adapter parameters.
2. `tp_size` is set to `1` -- PEFT is memory-efficient enough to run on
   a single GPU.
3. The learning rate is higher (`1e-4` vs `5e-6` for SFT) because we
   are only training the small adapter matrices.

Let's inspect the full configuration.

In [ ]:
from pathlib import Path

print(Path("llama_peft_config.yaml").read_text())

### LoRA Configuration Explained

| Parameter | Value | Description |
|---|---|---|
| `match_all_linear` | `true` | Apply LoRA adapters to all linear layers in the model |
| `dim` | `8` | Rank of the low-rank decomposition (higher = more capacity, more memory) |
| `alpha` | `32` | Scaling factor -- the effective learning rate scales as `alpha / dim` |
| `use_triton` | `true` | Use Triton kernels for fused LoRA operations (faster on NVIDIA GPUs) |

**Choosing `dim` and `alpha`:**

* `dim=8` is a good starting point for most tasks. Increase to 16 or 32
  for more complex tasks or larger models.
* `alpha` is typically set to `4 * dim`. The effective contribution of
  the adapter is scaled by `alpha / dim`, so `alpha=32, dim=8` gives a
  scaling factor of 4.

---
## Step 3. Launch PEFT

Since PEFT only trains a small number of adapter parameters, it fits
comfortably on a single GPU. No `--nproc-per-node` flag is needed
(defaults to 1).

In [ ]:
# PEFT works on a single GPU!
!automodel llama_peft_config.yaml

---
## Step 4. Generate from the PEFT Checkpoint

After training, the consolidated checkpoint contains the base model
weights merged with the LoRA adapters. It can be loaded with Hugging Face
`transformers` like any standard model.

In [ ]:
import glob
import os
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "meta-llama/Llama-3.2-1B"
CHECKPOINT_DIR = os.environ.get("CHECKPOINT_DIR", "checkpoints/krishnas_best")


def checkpoint_step(path: str) -> int:
    """Return the step number encoded in a checkpoint directory name."""
    return int(Path(path).name.rsplit("_step_", 1)[-1])


ckpt_dirs = sorted(glob.glob(f"{CHECKPOINT_DIR}/epoch_*_step_*"), key=checkpoint_step)
if not ckpt_dirs:
    print(f"WARNING: No checkpoints found under {CHECKPOINT_DIR}/. Falling back to the base model.")
    latest = BASE_MODEL
    tokenizer = AutoTokenizer.from_pretrained(latest)
    model = AutoModelForCausalLM.from_pretrained(latest, torch_dtype=torch.bfloat16, device_map="auto")
else:
    from peft import AutoPeftModelForCausalLM

    latest = str(Path(ckpt_dirs[-1]) / "model")
    tokenizer = AutoTokenizer.from_pretrained(latest)
    model = AutoPeftModelForCausalLM.from_pretrained(latest, torch_dtype=torch.bfloat16, device_map="auto")

print(f"Loading checkpoint: {latest}")

prompt = "Context: The Eiffel Tower is located in Paris, France. Question: Where is the Eiffel Tower?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(output[0], skip_special_tokens=True))

---
## Step 5. Compare SFT vs PEFT

| Aspect | Full SFT | PEFT (LoRA) |
|---|---|---|
| **Trainable parameters** | 100% of model | ~0.1--1% of model |
| **GPU memory** | High (full optimizer states) | Low (optimizer states for adapters only) |
| **Training speed** | Slower (full backward pass) | Faster (small adapter backward pass) |
| **Multi-GPU required** | Often yes for large models | Usually single GPU is sufficient |
| **Quality ceiling** | Highest for given data | Slightly lower, gap narrows with more data |
| **Adapter swapping** | Requires full model reload | Swap adapters without reloading base model |

**When to use which:**

* **Use full SFT** when you have ample GPU resources and want maximum
  quality, or when the task is very different from the pre-training
  distribution.
* **Use PEFT** when GPU memory is limited, you need to maintain multiple
  task-specific adapters, or you want faster iteration during
  experimentation.

---
## Next Steps

* **Full SFT** -- See the [SFT tutorial](sft.ipynb) for full-parameter
  fine-tuning.
* **Evaluation** -- Run the [Evaluation tutorial](../evaluation/) to
  benchmark your fine-tuned model.
* **RLHF** -- For reinforcement learning from human feedback (DPO, PPO),
  see [NeMo-RL](https://github.com/NVIDIA/NeMo-RL).

---
## Resources

* [NeMo AutoModel](https://github.com/NVIDIA-NeMo/Automodel) -- the
  training framework used in this tutorial.
* [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685) --
  the original LoRA paper.
* [NeMo Curator](https://github.com/NVIDIA/NeMo-Curator) -- tools for
  large-scale data curation and preprocessing.
* [NeMo-RL](https://github.com/NVIDIA/NeMo-RL) -- RLHF, DPO, and PPO
  alignment pipelines.
* [Hugging Face Transformers](https://huggingface.co/docs/transformers) --
  for checkpoint loading and inference.